In [ ]:
# ==========================================
# PHASE 3 : TEXT PREPROCESSING
# ==========================================

import pandas as pd
import numpy as np

import re
import string

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import warnings
warnings.filterwarnings("ignore")

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# Load processed dataset
df = pd.read_csv("../data/processed_reviews.csv")

print("Dataset Loaded Successfully!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shaik\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shaik\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\shaik\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Dataset Loaded Successfully!


In [2]:
df = df[["Text", "Sentiment"]]

df.head()

,Text,Sentiment
0,I have bought several of the Vitality canned d...,Positive
1,Product arrived labeled as Jumbo Salted Peanut...,Negative
2,This is a confection that has been around a fe...,Positive
3,If you are looking for the secret ingredient i...,Negative
4,Great taffy at a great price. There was a wid...,Positive


In [3]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english"))

In [4]:
def preprocess_ml(text):

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove numbers
    text = re.sub(r"\d+", " ", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Remove stopwords
    words = [
        word for word in text.split()
        if word not in stop_words
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)

In [6]:
df["Clean_Text"] = df["Text"].apply(preprocess_ml)

df.head()

,Text,Sentiment,Clean_Text
0,I have bought several of the Vitality canned d...,Positive,bought several vitality canned dog food produc...
1,Product arrived labeled as Jumbo Salted Peanut...,Negative,product arrived labeled jumbo salted peanutsth...
2,This is a confection that has been around a fe...,Positive,confection around century light pillowy citrus...
3,If you are looking for the secret ingredient i...,Negative,looking secret ingredient robitussin believe f...
4,Great taffy at a great price. There was a wid...,Positive,great taffy great price wide assortment yummy ...


In [7]:
def preprocess_bert(text):

    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(r"http\S+|www\S+", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [8]:
df["BERT_Text"] = df["Text"].apply(preprocess_bert)

df.head()

,Text,Sentiment,Clean_Text,BERT_Text
0,I have bought several of the Vitality canned d...,Positive,bought several vitality canned dog food produc...,I have bought several of the Vitality canned d...
1,Product arrived labeled as Jumbo Salted Peanut...,Negative,product arrived labeled jumbo salted peanutsth...,Product arrived labeled as Jumbo Salted Peanut...
2,This is a confection that has been around a fe...,Positive,confection around century light pillowy citrus...,This is a confection that has been around a fe...
3,If you are looking for the secret ingredient i...,Negative,looking secret ingredient robitussin believe f...,If you are looking for the secret ingredient i...
4,Great taffy at a great price. There was a wid...,Positive,great taffy great price wide assortment yummy ...,Great taffy at a great price. There was a wide...


In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["Label"] = label_encoder.fit_transform(df["Sentiment"])

df.head()

,Text,Sentiment,Clean_Text,BERT_Text,Label
0,I have bought several of the Vitality canned d...,Positive,bought several vitality canned dog food produc...,I have bought several of the Vitality canned d...,2
1,Product arrived labeled as Jumbo Salted Peanut...,Negative,product arrived labeled jumbo salted peanutsth...,Product arrived labeled as Jumbo Salted Peanut...,0
2,This is a confection that has been around a fe...,Positive,confection around century light pillowy citrus...,This is a confection that has been around a fe...,2
3,If you are looking for the secret ingredient i...,Negative,looking secret ingredient robitussin believe f...,If you are looking for the secret ingredient i...,0
4,Great taffy at a great price. There was a wid...,Positive,great taffy great price wide assortment yummy ...,Great taffy at a great price. There was a wide...,2


In [10]:
mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
)

print(mapping)

{'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}


In [12]:
import joblib

joblib.dump(
    label_encoder,
    "../models/label_encoder.pkl"
)

print("Label Encoder Saved Successfully!")

Label Encoder Saved Successfully!


In [13]:
df.to_csv(
    "../data/preprocessed_reviews.csv",
    index=False
)

print("Preprocessed Dataset Saved Successfully!")

Preprocessed Dataset Saved Successfully!
